# AL10 · Segment Anything vs a U-Net on the air photos
**MCS 3950 Computer Vision · UPEI · Fall 2026 · Active Learning 10 (Fri Nov 27)**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewgodbout/MCS-3950-Sandbox/blob/main/Notebooks/AL10-Segmentation/AL10_Segmentation.ipynb)

L16 showed two ways to get a **mask** (a label for every pixel):

* **prompt a foundation model**: SAM cuts out whatever you click on, with no training;
* **train a network** on labelled pixels: a U-Net.

Today you do both on one place in **1935, 1968 and 2020**, and score them against masks **drawn by hand**. That's
the check every project that uses SAM needs in its report.

**Pick a path** (set `PATH` in the first code cell). Both are new places, not project sites; the photos and the
methods are the same as the projects use.

| `PATH` | Place | What to segment |
|---|---|---|
| `"coast"` | Naufrage Harbour (Kings County) | **water**: harbour, estuary, river mouth, sea |
| `"wood"` | farmland in central Prince County | **woodland**: woodlots and treed stream strips |

**How the tasks work.** You won't write code today. Each task cell is fully commented out:
**click in the cell, press Ctrl+A then Ctrl+/** (Cmd+A, Cmd+/ on a Mac) to uncomment it, then run it. Your
work is the **🔮 predictions** (write them down *before* you run the cell) and the **questions**.

| Part | What | Time |
|---|---|---|
| 0 | Setup, the tiles and the hand-drawn reference | 5 min |
| 1 | **Task 1**: scoring a mask (IoU, precision, recall) | 7 min |
| 2 | One click per year | 4 min |
| 3 | **Task 2**: where you click | 7 min |
| 4 | **Task 3**: telling SAM more | 8 min |
| 4b | **Task 4**: "not this" clicks and a box, kept in a CSV file | 7 min |
| 5 | A U-Net trained on SAM's masks | 10 min |
| 6 | Wrap-up | 3 min |

**Runtime:** use a GPU if you can (Runtime → Change runtime type → T4 GPU). On a CPU the notebook still runs:
about 30 s to encode the three photos and about 1.5 min to train the U-Net.

## 0 · Setup

In [ ]:
PATH = "coast"          # "coast" (water at Naufrage Harbour) or "wood" (woodland in central Prince County)

import os, io, sys, json, time, zipfile, subprocess, urllib.request
os.environ["YOLO_VERBOSE"] = "False"
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from ultralytics.models.sam import SAM2Predictor

plt.rcParams.update({"figure.dpi": 100, "font.size": 11})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| device", DEVICE, "| path", PATH)

BASE = "https://raw.githubusercontent.com/andrewgodbout/MCS-3950-F26/main/Notebooks/data/"
if not os.path.exists("AL10/coast/reference.json"):
    with urllib.request.urlopen(BASE + "AL10.zip", timeout=180) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
    print("AL10.zip downloaded")

YEARS = (1935, 1968, 2020)
TILES = {y: cv2.imread(f"AL10/{PATH}/{y}." + ("jpg" if y == 2020 else "png"), cv2.IMREAD_COLOR) for y in YEARS}
INFO = json.load(open(f"AL10/{PATH}/reference.json"))

def polygons_to_mask(polys, holes=(), shape=(1024, 1024)):
    m = np.zeros(shape, np.uint8)
    for p in polys:
        cv2.fillPoly(m, [np.array(p, np.int32)], 1)
    for h in holes:
        cv2.fillPoly(m, [np.array(h, np.int32)], 0)
    return m.astype(bool)

REF = {y: polygons_to_mask(INFO["years"][str(y)]["polygons"], INFO["years"][str(y)]["holes"]) for y in YEARS}
WHAT = "water" if PATH == "coast" else "woodland"
print(INFO["site"])
for y in YEARS:
    print(f"{y}: tile {TILES[y].shape[1]} x {TILES[y].shape[0]}, reference {WHAT} = {REF[y].mean():.0%} of the tile")

Each tile is **1024 × 1024 pixels at 1 m per pixel** (about 1 × 1 km), north up, and the three years share one
grid: pixel (x, y) is the same spot on the ground in every year. 1935 and 1968 are grey; 2020 is colour.

The **reference** (red outline below) was drawn by hand, by one person, as polygons on zoomed-in copies with a grid.
It follows a written **rule** (printed below). Expect its edges to be off by about 5 pixels, more on blurry 1935
edges. A reference is not perfect; it is a careful, *independent* answer to check SAM against.

In [ ]:
def rgb(img):
    return img[..., ::-1]

def outline(ax, mask, color="red", lw=1.2):
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    for c in cs:
        if len(c) > 20:
            ax.plot(c[:, 0, 0], c[:, 0, 1], color=color, lw=lw)

print("Rule:", INFO["rule"])
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
for ax, y in zip(axes, YEARS):
    ax.imshow(rgb(TILES[y])); outline(ax, REF[y]); ax.set_title(f"{y}: reference {WHAT} (red)"); ax.axis("off")
plt.tight_layout(); plt.show()

Now SAM (SAM 2.1 "base", 160 MB, downloaded from GitHub the first time). SAM works in two steps: a big network
**encodes** a photo once (the slow part), then a small one turns each **prompt** (click or box) into a mask plus a
**score**, SAM's own guess of how good the mask is (0–1). We encode the three photos once now and keep the results,
so every prompt after this takes a fraction of a second.

In [ ]:
predictor = SAM2Predictor(overrides=dict(model="sam2.1_b.pt", task="segment", mode="predict",
                                         imgsz=1024, conf=0.0, save=False, verbose=False))
FEATS = {}
for y in YEARS:
    t0 = time.time(); predictor.set_image(TILES[y]); FEATS[y] = predictor.features
    print(f"encoded {y} in {time.time() - t0:.1f} s")

def use(year):
    '''Point SAM at one year's photo (re-uses the stored encoding).'''
    predictor.setup_source(TILES[year]); predictor.features = FEATS[year]

def segment(points, labels, box=None):
    '''ONE object from one prompt. points: [(x, y), ...]; labels: 1 = "this", 0 = "not this".
    Returns (mask: 1024 x 1024 bool array, score).'''
    kw = dict(points=[[list(map(int, p)) for p in points]], labels=[list(labels)])
    if box is not None:
        kw["bboxes"] = [list(map(int, box))]
    r = predictor(**kw)[0]
    return r.masks.data[0].cpu().numpy().astype(bool), float(r.boxes.conf[0])

GREEN, RED, BLUE = np.array([0, 200, 0]), np.array([230, 30, 30]), np.array([40, 90, 255])

def error_map(ax, year, pred, title="", points=None):
    '''green = right (in both), red = predicted but not in the reference, blue = in the reference but missed.'''
    show = rgb(TILES[year]).astype(float)
    ref = REF[year]
    for m, col in ((pred & ref, GREEN), (pred & ~ref, RED), (~pred & ref, BLUE)):
        show[m] = 0.45 * show[m] + 0.55 * col
    ax.imshow(show.astype(np.uint8)); ax.set_title(title, fontsize=10); ax.axis("off")
    for p in points or []:
        ax.plot(*p, marker="*", ms=16, color="yellow", mec="black")

mask_scores = None      # Task 1 defines this

## 1 · Task 1: scoring a mask (≈ 7 min)

Three numbers, all from counting pixels. Call the reference **R** and the predicted mask **P**:

* **TP** (true positive): in P and in R. **FP**: in P, not in R. **FN**: in R, missed by P.
* **IoU** = TP / (TP + FP + FN): the overlap divided by everything either one calls water/woodland (L15, L16).
* **Precision** = TP / (TP + FP): of what P marked, how much is right? *Low precision = it grabbed too much.*
* **Recall** = TP / (TP + FN): of what is really there, how much did P find? *Low recall = it missed some.*

**By hand.** A 5 × 5 image. `R` = reference, `P` = prediction (1 = water):

```
      R                 P
0 0 0 0 0         0 0 0 0 0
0 1 1 1 0         0 1 1 0 0
0 1 1 1 0         0 1 1 1 0
0 0 1 1 0         0 0 0 1 1
0 0 0 0 0         0 0 0 0 0
```

R has 8 pixels and P has 7; they share **6**. So TP = 6, FP = 1, FN = 2:
IoU = 6 / 9 = **0.67**, precision = 6 / 7 = **0.86**, recall = 6 / 8 = **0.75**.

🔮 **Predict** before you run the check cell: what are IoU, precision and recall for

* (a) a "mask" that says **everything** (all 25 pixels) is water?
* (b) a mask with just **one** pixel, and that pixel is inside R?

*Your predictions (a) and (b):*

**Task 1.** Uncomment (Ctrl+A, then Ctrl+/) and run.

In [ ]:
## ▼ Task 1: select the whole cell (Ctrl+A) and press Ctrl+/ (Cmd+A, Cmd+/ on a Mac), then run it.
# def mask_scores(pred, ref):
#     '''pred, ref: boolean arrays of the same shape -> dict with IoU, precision and recall.'''
#     tp = (pred & ref).sum()                  # predicted, and really there
#     fp = (pred & ~ref).sum()                 # predicted, but not there
#     fn = (~pred & ref).sum()                 # there, but missed
#     return dict(IoU=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))

In [ ]:
if mask_scores is None:
    print("Uncomment Task 1 first.")
else:
    R5 = np.array([[0,0,0,0,0],[0,1,1,1,0],[0,1,1,1,0],[0,0,1,1,0],[0,0,0,0,0]], bool)
    P5 = np.array([[0,0,0,0,0],[0,1,1,0,0],[0,1,1,1,0],[0,0,0,1,1],[0,0,0,0,0]], bool)
    one = np.zeros((5, 5), bool); one[2, 2] = True
    for name, p in (("by-hand P", P5), ("(a) everything", np.ones((5, 5), bool)), ("(b) one pixel", one)):
        s = mask_scores(p, R5)
        print(f"{name:15s} IoU {s['IoU']:.2f}   precision {s['precision']:.2f}   recall {s['recall']:.2f}")

**Q1.** A mask can have perfect recall and poor IoU, or perfect precision and poor IoU. Which number tells you
SAM **grabbed too much**, and which tells you it **missed a piece**? Why report all three, not just IoU?

*Your answer:*

## 2 · One click per year (≈ 4 min, nothing to uncomment)

One positive click (yellow star), at the same ground position in all three years.

🔮 **Predict:** which year will get the **lowest IoU**, and will that be because of low precision or low recall?

*Your prediction:*

In [ ]:
PARTS = {"coast": {"harbour": (790, 420), "sea": (880, 40), "estuary bend": (700, 660), "lower river": (350, 850)},
         "wood":  {"big woods (SE)": (880, 900), "woodlot (W)": (100, 640), "woods (SW)": (150, 960),
                   "stream strip": (570, 450), "strip by the clearing": (505, 680)}}[PATH]
CLICK = list(PARTS.values())[0]

if mask_scores is None:
    print("Uncomment Task 1 first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
    print(f"one click at {CLICK} ({list(PARTS)[0]})")
    for ax, y in zip(axes, YEARS):
        use(y); m, sc = segment([CLICK], [1]); s = mask_scores(m, REF[y])
        print(f"{y}: SAM score {sc:.2f} | IoU {s['IoU']:.2f}  precision {s['precision']:.2f}  recall {s['recall']:.2f}")
        error_map(ax, y, m, f"{y}: IoU {s['IoU']:.2f}, score {sc:.2f}", [CLICK])
    plt.tight_layout(); plt.show()
    print("green = right, red = SAM's mask but not in the reference, blue = in the reference but missed")

**Q2.** Look at the blue (missed) areas. Is one click a fair test of SAM here? What does a single click
*ask* SAM for?

*Your answer:*

## 3 · Task 2: where you click (≈ 7 min)

In your project you'll click by eye. How much does the exact spot matter? Here: **12 random clicks**, every one
of them **inside** the reference, one prompt each, in each year. Each click becomes a dot at (recall, precision).

🔮 **Predict:** will the 12 masks be about the same? Will any of them have **precision below 0.5** (more wrong
than right)?

*Your prediction:*

In [ ]:
## ▼ Task 2: select the whole cell (Ctrl+A) and press Ctrl+/ (Cmd+A, Cmd+/ on a Mac), then run it.
# rng = np.random.default_rng(0)
# JITTER = {}                                                   # year -> list of (x, y, score, scores dict)
# for y in YEARS:
#     use(y)
#     ys, xs = np.nonzero(REF[y])                               # every pixel inside the reference
#     pick = rng.choice(len(xs), 12, replace=False)             # 12 of them at random
#     JITTER[y] = []
#     for i in pick:
#         m, sc = segment([(xs[i], ys[i])], [1])
#         JITTER[y].append((xs[i], ys[i], sc, mask_scores(m, REF[y])))

In [ ]:
if "JITTER" not in globals():
    print("Uncomment Task 2 first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    cols = {1935: "tab:brown", 1968: "tab:gray", 2020: "tab:green"}
    for y in YEARS:
        r = [s["recall"] for *_, s in JITTER[y]]; p = [s["precision"] for *_, s in JITTER[y]]
        axes[0].scatter(r, p, s=60, color=cols[y], label=str(y), alpha=0.8, edgecolor="k")
        iou = [s["IoU"] for *_, s in JITTER[y]]
        bad = sum(pp < 0.5 for pp in p)
        print(f"{y}: IoU median {np.median(iou):.2f} (range {min(iou):.2f}-{max(iou):.2f}) | "
              f"recall median {np.median(r):.2f} | clicks with precision < 0.5: {bad} of 12")
    axes[0].set_xlabel("recall (how much of the reference it found)"); axes[0].set_ylabel("precision (how much of the mask is right)")
    axes[0].set_xlim(-0.02, 1.02); axes[0].set_ylim(-0.02, 1.05); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[0].set_title("12 random clicks inside the reference, per year")
    y = 1968
    axes[1].imshow(rgb(TILES[y])); outline(axes[1], REF[y], "red")
    for x0, y0, sc, s in JITTER[y]:
        axes[1].plot(x0, y0, "o", ms=9, color="yellow" if s["precision"] >= 0.5 else "magenta", mec="k")
        axes[1].text(x0 + 12, y0, f"{s['recall']:.2f}", color="white", fontsize=8, weight="bold")
    axes[1].set_title(f"{y}: click positions, labelled with recall (magenta = precision < 0.5)"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

**Q3.** Describe the cloud of dots. Why are most recalls well below 1 even when precision is near 1? What happened
at a click with precision below 0.5 (look at where it is)?

*Your answer:*

## 4 · Task 3: telling SAM more (≈ 8 min)

You have one click per **part** (`PARTS`, printed below). Four ways to use them:

* **A**: just the first click (§2).
* **B**: all the clicks in **one** prompt.
* **C**: **one prompt per click**, then join (union) the masks.
* **D**: like C, but leave out any mask whose SAM score is below 0.5.

🔮 **Predict:** B or C, which is better, and why might they differ? Will D (dropping low-score masks) help?

*Your prediction:*

In [ ]:
for name, p in PARTS.items():
    print(f"{name:22s} click at {p}")

In [ ]:
## ▼ Task 3: select the whole cell (Ctrl+A) and press Ctrl+/ (Cmd+A, Cmd+/ on a Mac), then run it.
# pts = list(PARTS.values())
# WAYS, SAM_C, PER_CLICK = {}, {}, {}
# for y in YEARS:
#     use(y)
#     a, _ = segment(pts[:1], [1])                              # A: one click
#     b, _ = segment(pts, [1] * len(pts))                       # B: all clicks in ONE prompt = "one object covering all of these"
#     each = [segment([p], [1]) for p in pts]                   # one prompt per click: a list of (mask, score)
#     c = np.any([m for m, s in each], axis=0)                  # C: union of all of them
#     d = np.any([m for m, s in each if s >= 0.5] or [np.zeros_like(c)], axis=0)    # D: union of the confident ones
#     WAYS[y] = {"A": a, "B": b, "C": c, "D": d}
#     SAM_C[y] = c                                              # kept for the U-Net in part 5
#     PER_CLICK[y] = [(s, m.mean()) for m, s in each]           # score and size of each click's mask

In [ ]:
if "WAYS" not in globals():
    print("Uncomment Task 3 first.")
else:
    print("IoU with the reference (precision / recall)")
    print("year" + "".join(f"  {'way ' + w:^16s}" for w in "ABCD"))
    for y in YEARS:
        row = ""
        for w in "ABCD":
            s = mask_scores(WAYS[y][w], REF[y]); row += f"  {s['IoU']:.2f} ({s['precision']:.2f}/{s['recall']:.2f})"
        print(f"{y}{row}")
    print("\nper-click SAM score and mask size (fraction of the tile):")
    for y in YEARS:
        print(f"{y}: " + "  ".join(f"{n}: {s:.2f}, {a:.0%}" for n, (s, a) in zip(PARTS, PER_CLICK[y])))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
    for ax, y in zip(axes, YEARS):
        s = mask_scores(WAYS[y]["C"], REF[y])
        error_map(ax, y, WAYS[y]["C"], f"{y}, way C: IoU {s['IoU']:.2f}", list(PARTS.values()))
    plt.tight_layout(); plt.show()

**Q4.** Which way is best in each year? Explain B vs C in your own words. Did D help or hurt, and what does that
tell you about trusting SAM's score?

*Your answer:*

## 4b · Task 4: "not this" clicks and a box, kept in a file (≈ 7 min)

Two kinds of mistake look the same in an IoU table but need different fixes:

* **Spill**: SAM's mask floods from the part you meant into a neighbour (a click on a narrow strip that takes the
  field beside it). Fix: **negative clicks** (label 0, "not this") on the neighbour, and/or a **box** round the part.
* **Disagreement with your rule**: SAM sees one region where your rule draws a line (tidal flat vs water, scrub vs
  woods). "Not this" clicks often can't move that line; you have to edit the mask or change the rule.

**Way E** = way C, plus negative clicks (and a box) on the part that went wrong. The prompts are written to a
**CSV file**, and the masks are made **from the file**. That's how to keep prompts in your project: the same
clicks every year, re-runnable by anyone, and handed in with your results. One row per click or box:
`part, kind (pos / neg / box), x, y, x2, y2` (x2, y2 only for a box: its bottom-right corner).

🔮 **Predict:** on your path, will way E beat way C? In which year?

*Your prediction:*

In [ ]:
import csv

PROMPTS_E = {
    "coast": [("harbour", "pos", 790, 420), ("harbour", "neg", 583, 535), ("harbour", "neg", 600, 560),
              ("sea", "pos", 880, 40),
              ("estuary bend", "pos", 700, 660), ("estuary bend", "neg", 583, 535), ("estuary bend", "neg", 600, 560),
              ("lower river", "pos", 350, 850)],
    "wood":  [("big woods (SE)", "pos", 880, 900), ("woodlot (W)", "pos", 100, 640), ("woods (SW)", "pos", 150, 960),
              ("stream strip", "pos", 570, 450), ("stream strip", "neg", 400, 300), ("stream strip", "neg", 750, 300),
              ("stream strip", "box", 460, 0, 660, 560),
              ("strip by the clearing", "pos", 505, 680)],
}[PATH]
PROMPT_FILE = f"al10_prompts_{PATH}.csv"

def write_prompts(filename, rows):
    with open(filename, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["part", "kind", "x", "y", "x2", "y2"])
        for r in rows:
            w.writerow(list(r) + [""] * (6 - len(r)))

def read_prompts(filename):
    # CSV -> {part: {"points": [(x, y), ...], "labels": [1 or 0, ...], "box": (x1, y1, x2, y2) or None}}, in file order
    parts = {}
    for r in csv.DictReader(open(filename)):
        p = parts.setdefault(r["part"], {"points": [], "labels": [], "box": None})
        if r["kind"] == "box":
            p["box"] = (int(r["x"]), int(r["y"]), int(r["x2"]), int(r["y2"]))
        else:
            p["points"].append((int(r["x"]), int(r["y"]))); p["labels"].append(1 if r["kind"] == "pos" else 0)
    return parts

write_prompts(PROMPT_FILE, PROMPTS_E)
print(open(PROMPT_FILE).read())

In [ ]:
## ▼ Task 4: select the whole cell (Ctrl+A) and press Ctrl+/ (Cmd+A, Cmd+/ on a Mac), then run it.
# PROMPTS = read_prompts(PROMPT_FILE)                           # the masks come from the file, not from code
# WAY_E, PER_PART_E = {}, {}
# for y in YEARS:
#     use(y)
#     each = [segment(p["points"], p["labels"], p["box"]) for p in PROMPTS.values()]   # one prompt per part
#     WAY_E[y] = np.any([m for m, s in each], axis=0)           # E: union, as in way C
#     PER_PART_E[y] = [(s, m.mean()) for m, s in each]

In [ ]:
if "WAY_E" not in globals() or "WAYS" not in globals():
    print("Uncomment Tasks 3 and 4 first.")
else:
    print("IoU with the reference (precision / recall)")
    print("year" + "".join(f"  {'way ' + w:^16s}" for w in "CE"))
    for y in YEARS:
        c, e = mask_scores(WAYS[y]["C"], REF[y]), mask_scores(WAY_E[y], REF[y])
        print(f"{y}  {c['IoU']:.2f} ({c['precision']:.2f}/{c['recall']:.2f})  {e['IoU']:.2f} ({e['precision']:.2f}/{e['recall']:.2f})")
    print("\nway E, per-part SAM score and mask size:")
    for y in YEARS:
        print(f"{y}: " + "  ".join(f"{n}: {s:.2f}, {a:.0%}" for n, (s, a) in zip(PROMPTS, PER_PART_E[y])))
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    for row, (name, masks) in enumerate((("C", {y: WAYS[y]["C"] for y in YEARS}), ("E", WAY_E))):
        for ax, y in zip(axes[row], YEARS):
            s = mask_scores(masks[y], REF[y])
            error_map(ax, y, masks[y], f"{y}, way {name}: IoU {s['IoU']:.2f}")
            if name == "E":
                for p in PROMPTS.values():
                    for (x, yy), l in zip(p["points"], p["labels"]):
                        ax.plot(x, yy, marker="*" if l else "X", ms=14, color="yellow" if l else "red", mec="black")
                    if p["box"]:
                        x1, y1, x2, y2 = p["box"]
                        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", lw=2))
    plt.tight_layout(); plt.show()
    print("yellow star = positive click, red X = negative click, yellow rectangle = box")

**Q4b.** Where did way E help, and where didn't it? For each red patch left in way E's maps, say whether it is a
**spill** or a **disagreement with the rule**, and what you would do about it in a project.

*Your answer:*

**Q5.** Your project report has to say how good its SAM masks are. Using the tables in §4 and §4b, write the
sentence you would put in a report about SAM on **1935** photos.

*Your answer:*

## 5 · A U-Net trained on SAM's masks (≈ 10 min, nothing to uncomment)

A tempting project shortcut: *let SAM label one year, train a U-Net on those masks, then run the U-Net on every
year.* Here: L16's small U-Net, trained on the **1968** tile with SAM's way-C mask as the label (no hand
labels at all), then run on 1935, 1968 and 2020.

* Input: the photo in grey (2020 is converted), with each photo **standardised** (mean 0, standard deviation 1),
  so an overall brighter or darker print looks the same to the network.
* Training: 200 steps, each on 6 random 192 × 192 crops of the 1968 tile, flipped and turned (L14's augmentation),
  with small random brightness/contrast changes.

🔮 **Predict** before you run it:

* (a) On **1968**, the year it learned from, will the U-Net's IoU against the **reference** be higher or lower
  than SAM's?
* (b) On **1935** and **2020**, will it do better or worse than on 1968?

*Your predictions (a) and (b):*

In [ ]:
def block(cin, cout):                      # two 3x3 conv layers with batch norm and ReLU (L13, L16)
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())

class UNet(nn.Module):                     # L16's U-Net: 4 pools down, 4 transposed convs up, skip connections
    def __init__(self, c=8):
        super().__init__()
        self.d1, self.d2, self.d3, self.d4 = block(1, c), block(c, 2*c), block(2*c, 4*c), block(4*c, 8*c)
        self.bottom = block(8*c, 16*c)
        self.up4, self.c4 = nn.ConvTranspose2d(16*c, 8*c, 2, 2), block(16*c, 8*c)
        self.up3, self.c3 = nn.ConvTranspose2d(8*c, 4*c, 2, 2), block(8*c, 4*c)
        self.up2, self.c2 = nn.ConvTranspose2d(4*c, 2*c, 2, 2), block(4*c, 2*c)
        self.up1, self.c1 = nn.ConvTranspose2d(2*c, c, 2, 2), block(2*c, c)
        self.score = nn.Conv2d(c, 2, 1)
    def forward(self, x):
        x1 = self.d1(x); x2 = self.d2(F.max_pool2d(x1, 2)); x3 = self.d3(F.max_pool2d(x2, 2))
        x4 = self.d4(F.max_pool2d(x3, 2)); b = self.bottom(F.max_pool2d(x4, 2))
        y = self.c4(torch.cat([self.up4(b), x4], 1)); y = self.c3(torch.cat([self.up3(y), x3], 1))
        y = self.c2(torch.cat([self.up2(y), x2], 1)); y = self.c1(torch.cat([self.up1(y), x1], 1))
        return self.score(y)

def grey(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    return (g - g.mean()) / g.std()                                   # standardise each photo

def train_unet(x, label, steps=200, batch=6, size=192, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = UNet().to(DEVICE); opt = torch.optim.Adam(net.parameters(), 2e-3); t0 = time.time()
    for it in range(steps):
        xs, ys = [], []
        for _ in range(batch):
            r0, c0 = rng.integers(0, x.shape[0] - size, 2)
            xi, yi = x[r0:r0+size, c0:c0+size], label[r0:r0+size, c0:c0+size]
            if rng.random() < 0.5: xi, yi = xi[:, ::-1], yi[:, ::-1]
            k = rng.integers(4); xi, yi = np.rot90(xi, k), np.rot90(yi, k)
            xi = xi * rng.uniform(0.7, 1.3) + rng.uniform(-0.3, 0.3)    # brightness / contrast jitter
            xs.append(xi.copy()); ys.append(yi.copy())
        xb = torch.tensor(np.array(xs))[:, None].to(DEVICE); yb = torch.tensor(np.array(ys)).long().to(DEVICE)
        loss = F.cross_entropy(net(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 50 == 0 or it == steps - 1:
            print(f"  step {it:3d}  loss {loss.item():.3f}  ({time.time() - t0:.0f} s)")
    return net

def predict(net, img):
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(torch.tensor(grey(img))[None, None].to(DEVICE)), 1)[0, 1].cpu().numpy() > 0.5

if "SAM_C" not in globals():
    print("Uncomment Task 3 first: the U-Net learns from SAM's way-C mask.")
else:
    net = train_unet(grey(TILES[1968]), SAM_C[1968].astype(np.int64))
    UNET = {y: predict(net, TILES[y]) for y in YEARS}

In [ ]:
if "UNET" in globals():
    print(f"{'':6s}{'SAM (way C)':>22s}{'U-Net':>22s}     (IoU, precision / recall, all against the reference)")
    for y in YEARS:
        s, u = mask_scores(SAM_C[y], REF[y]), mask_scores(UNET[y], REF[y])
        print(f"{y}    {s['IoU']:.2f} ({s['precision']:.2f}/{s['recall']:.2f})      "
              f"{u['IoU']:.2f} ({u['precision']:.2f}/{u['recall']:.2f})")
    print(f"\n1968: IoU between the U-Net's mask and SAM's mask (its own training label) = "
          f"{mask_scores(UNET[1968], SAM_C[1968])['IoU']:.2f}")
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
    for ax, y in zip(axes, YEARS):
        error_map(ax, y, UNET[y], f"{y}: U-Net trained on SAM's 1968 mask, IoU {mask_scores(UNET[y], REF[y])['IoU']:.2f}")
    plt.tight_layout(); plt.show()

**Q6.** Explain (a): why can't a network trained on SAM's masks be expected to beat SAM, *when both are scored
against the hand reference*? What would you be measuring if you scored the U-Net against SAM's masks instead?

*Your answer:*

**Q7.** Explain (b) using the error maps (red = too much, blue = missed) and precision/recall. Then compare with a
neighbour who took the **other** path: did the U-Net carry over between years equally well for water and for
woodland? Why might that be?

*Your answer:*

## 6 · Wrap-up: the recipe for your project

1. **Write the rule** for what counts (tidal flats? scrub? hedgerows?) before you label anything.
2. **Draw 10–20 hand masks** spread over the years, following the rule. Don't use SAM to make them.
3. **Prompt with a plan**: one prompt per part (way C), with "not this" clicks or a box only where a mask spills
   (way E), and the same prompts every year. Keep them in a **CSV file** like `al10_prompts_*.csv`, make the masks
   from the file, and hand the file in with your results.
4. **Report IoU, precision and recall per year** against your hand masks. If 1935 is worse, say so and say what
   it means for your conclusions.
5. **SAM's masks are not ground truth.** A model trained on them can only learn SAM's answer, mistakes included,
   so test it against the hand masks too.

**R1.** Which step of the recipe is your group missing right now, and who will do it?

*Your answer:*

## 7 · Directions to try in your project (optional, read after class)

Ideas for making SAM masks with fewer clicks across many tiles and years, and for catching bad masks
automatically. Each one was tried on the AL10 tiles (one tile per path, so treat the numbers as a hint, not a
promise for your sites). IoU is against the hand reference, 1935 / 1968 / 2020.

**1. Carry the clicks across years (water: worked).** Prompt one year by hand (here 1968, 4 clicks, checked
on the error map). Take SAM's mask, pick 8 points **deep inside** it (the peaks of
`cv2.distanceTransform(mask)`, spread out), and use those points as one-click prompts in every other year:
the years share one pixel grid. Coast: **0.71 / 0.79 / 0.80**, better than the hand clicks (0.68 / 0.78 / 0.74),
because 8 seeds reach more stretches of water than 4. Woodland: 0.23 / 0.85 / 0.80: seeds from 1968's woods
landed on 1935 fields, because the woods **grew**. Carried clicks assume the thing is mostly where it was, and
they never find anything new (a pond dug after the starting year).

**2. Area check across years (water: a good flag).** Coastlines change slowly, so a year whose water area jumps
is suspect. One spilled click made 1935 **+29%** bigger than 1968; the hand references themselves differ by at
most 8% between years. A rule like "flag any part more than ~15% off its neighbouring years" would catch it.
Check **each part** as well as the total (a spill and a miss can cancel), only **flag** (never throw masks out
automatically, or you hide the change you're measuring), and remember it can't see a mistake made the same way
every year (SAM includes the tidal flat in all three years). That's what your hand-drawn test masks are for.

**3. Negative clicks: by hand, where a check flags a spill.** Automatic negatives placed in a ring just outside
the starting year's mask did **not** help (coast 0.70 / 0.78 / 0.79 at 10 px out; woodland 2020 fell to 0.60):
the edge is exactly where things change between years, so some negatives land on water or woods in other years.
They also can't reach the tidal flat, which is *inside* SAM's mask. Many negatives on every click made SAM's
scores collapse. Use way E (Task 4): look at the flagged mask, put one or two negatives on the spill itself.

**4. Woodland: let a U-Net carry it across years.** Woodland is recognised by **texture**, which survives the
change of film and colour. The U-Net trained on SAM's 1968 mask (part 5) scored **0.68 / 0.85 / 0.84**, far
better than any SAM clicks in 1935. Using its prediction to place SAM clicks in each year helped only slightly in
1968 (0.88) and not in 1935 (0.35). For water the opposite holds: water is recognised by **tone**, which changes
from print to print (part 5's 0.29 / 0.73 / 0.39), so carry clicks (idea 1) instead.

**5. Fields: check structure, not area.** In farm consolidation, change is the signal, so there is no stable area
to check against. Check what should be true instead: field **counts fall and sizes grow** over the years;
two fields **merging** is expected, one field **splitting** in two between years is suspect (flag it); roads and
long-lasting hedgerows are stable lines that field edges should follow. Count fields by hand on a few sample
tiles per year as your test set.

**6. Refining the edges with a U-Net (tried: no gain here).** Idea: keep the carried SAM masks (idea 1), but let a
U-Net re-decide only a narrow band (10 or 25 px) along each mask's edge. Two versions: the 1968 U-Net from part 5,
or a fresh U-Net per year trained on that year's own photo, using the confident inside and outside of its SAM
mask as labels and ignoring the band. Best result 0.73 / 0.67 / 0.72 (own-year U-Net, 10 px band) against
0.71 / 0.79 / 0.80 without it; most versions were worse. Why: SAM's edges already follow the edges in the photo,
and the remaining errors are **whole regions** (the tidal flat, unprompted channels), not edges a few metres off.
Edge errors smaller than about 5 m are also below what the hand references and the alignment between years can
measure. Where it might be worth a try: masks made from a **whole** 3400 × 3000 tile, which SAM sees at about
3.3 m per pixel; but running SAM on 1024 × 1024 crops (§9 of the SAM helper) is the simpler fix for that.

A third version learns from your **hand-drawn** edges instead of SAM's: train the U-Net on patches cut along your
hand-drawn coastline (water on one side, land on the other), so it learns your **rule** (e.g. "the tidal flat is
land"), then let it re-decide the band. On the AL10 tile it only helped when trained on the other half of the
**same year** (1935: 0.71 → 0.76); trained on other years it was worse (0.64 / 0.70 / 0.69), and on its own it
called far too much water. One tile is too little coastline to learn from, tone changes between years, and the
tidal flat appears in only one place. To give it a fair chance: hand-drawn edge strips from **several sites**
(strips of coast are quicker to draw than whole masks), a **pretrained** encoder (L14, AL8) rather than training
from scratch, and a whole site held out for testing. Keep those test masks out of training.

*Which pretrained encoder?* AL8's **ImageNet ResNet-18**: it already worked on these grey photos, it is small enough
for a few kilometres of hand-drawn coast, and its weights load the way AL8 loaded them. Prepare the input as in AL8
Task 1 (grey to 3 channels, ImageNet mean/std), use L16's decoder with skips from ResNet's stages, and train in
AL8's order (new decoder first, then `layer4` at a low learning rate). The `segmentation_models_pytorch` package
builds the same network in one line: `smp.Unet("resnet18", encoder_weights="imagenet", in_channels=1)` (check that
its weights download in Colab first). Avoid models pretrained on multispectral satellite images (about 10 m per pixel,
unlike these 1 m grey photos) and SAM's own encoder (its 64 × 64 grid is one cell per 16 m: too coarse for edges).

**Whatever you try:** keep your prompts in a CSV file, keep 10–20 hand-drawn test masks spread over the years, and
report IoU, precision and recall per year against them.

## Bonus A · Your own reference (inter-annotator check)

Two careful people never draw the same polygon. Below is a 256 × 256 window with a grid. Edit `MY_POLY` (x, y
pairs in **tile** pixels, going round the shape) to trace the edge of the water/woodland **inside the window**
by the rule, then run the cell. Compare your IoU with the reference against SAM's: if SAM is about as close to
the reference as you are, the remaining gap is label noise, not SAM's fault.

In [ ]:
WIN = {"coast": (600, 560), "wood": (440, 400)}[PATH]           # top-left (x, y) of the 256 x 256 window
YEAR_B = 1968
MY_POLY = {"coast": [(600, 600), (700, 580), (760, 640), (720, 720), (650, 780), (600, 790)],   # a rough start: edit me
           "wood":  [(500, 400), (560, 400), (600, 520), (620, 656), (500, 656), (480, 520)]}[PATH]

x0, y0 = WIN; win = (slice(y0, y0 + 256), slice(x0, x0 + 256))
mine = polygons_to_mask([MY_POLY]) & polygons_to_mask([[(x0, y0), (x0 + 256, y0), (x0 + 256, y0 + 256), (x0, y0 + 256)]])
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb(TILES[YEAR_B])[win], extent=(x0, x0 + 256, y0 + 256, y0))
for v in range(0, 257, 32):
    ax.axvline(x0 + v, color="yellow", lw=0.5); ax.axhline(y0 + v, color="yellow", lw=0.5)
ax.set_xticks(range(x0, x0 + 257, 32)); ax.set_yticks(range(y0, y0 + 257, 32)); ax.tick_params(labelsize=8)
outline(ax, REF[YEAR_B], "red"); ax.plot(*np.array(MY_POLY + MY_POLY[:1]).T, color="cyan", lw=2)
ax.set_xlim(x0, x0 + 256); ax.set_ylim(y0 + 256, y0); ax.set_title("reference (red) vs yours (cyan)"); plt.show()
if mask_scores is not None:
    print(f"inside the window: IoU(yours, reference) = {mask_scores(mine[win], REF[YEAR_B][win])['IoU']:.2f}")
    if "SAM_C" in globals():
        print(f"                   IoU(SAM way C, reference) = {mask_scores(SAM_C[YEAR_B][win], REF[YEAR_B][win])['IoU']:.2f}")

## Bonus B · The other path

Change `PATH` in the first code cell and run everything again (Runtime → Run all). Which of your answers change?